# 성주 개인 통합 정리 Notebook

### 작성 메타
- 프로젝트명: 성남시 젠트리피케이션 위험도 분석 시스템 구축 및 상생 체계 제안
- 담당자: 성주
- 담당 파트: 교통(지하철·버스), 신용(대민개방·전입·전출), 공시지가
- 작성일: 2026-04-29
- 목적: 각 데이터의 1차/2차 전처리 결과와 예외 처리 기준을 한 노트북에 통합 정리하여 팀 공유 가능한 재현 가능한 형태로 만든다.
- 최종 산출물: `2차전처리_지하철데이터.csv`, `2차전처리_버스데이터.csv`, `2차전처리_대민개방데이터.csv`, `2차전처리_전입데이터.csv`, `2차전처리_전출데이터.csv`, `2차전처리_공시지가데이터.csv` + 예외 처리 기준표

> 핵심 산출물은 "예외처리 기준의 명문화"이다. 특히
> - 버스 정류소-행정동 보정표,
> - 산 필지 제외 기준,
> - 신용 데이터 0값(triple_zero) 분류 기준
> 은 표로 같이 남겨야 팀이 재현할 수 있다.


## 1. 작업 개요

- 내가 맡은 데이터: 교통(지하철/버스), 신용(대민개방·전입·전출), 공시지가
- 왜 필요한지: 분석 결과보다 "어떤 값을 어떻게 처리했는지"가 재현성에 직접 영향을 주기 때문. 특히 0값과 행정동 매핑은 모든 조인 단계에서 영향을 준다.
- 최종적으로 남길 표/파일:
  - 문제 데이터 발견 목록 표
  - 정류소-행정동 보정표 (구체적 정류소ID 포함)
  - 산/공원/비주거성 필지 제외 기준
  - 신용 데이터 0값 분류 기준(structural_zero / error_zero)
  - clean 전후 비교 결과
  - 교통 EDA 핵심 해석 요약


## 2. 파일 정리 + 입력 파일 목록

### 작업 파일 정리표
| 파일명 | 현재 역할 | 유지 여부 | 비고 |
|---|---|---|---|
| (삭제 예정)1차 전처리.ipynb | 초기 작업본 | 삭제 예정 | 백업만 보관 |
| 2차 전처리_교통.ipynb | 교통(지하철/버스) 처리 | 유지 | 정류소-행정동 보정표 포함 |
| 2차 전처리_신용.ipynb | 신용/전입/전출 처리 | 유지 | triple_zero 분류 기준 포함 |
| 2차 전처리_공시지가.ipynb | 공시지가 처리 | 유지 | 산 필지 제외 |
| EDA_교통.ipynb | 교통 EDA | 유지 | HHI / 로그 변화율 / 사분면 분석 결과 |


### 입력 원본 파일 목록 (사용한 실제 파일)
| 파일명 | 설명 | 사용 여부 | 비고 |
|---|---|---|---|
| ../Data/subway.csv | 지하철 월별 승하차 | 사용 | 결측 0건 |
| ../Data/bus_plz.csv | 버스 정류소 일별 승하차 | 사용 | 행정동 65,110건 결측 → 매핑 보정 후 0건 |
| ../Data/신용정보.csv | 행정동/연령별 신용 인구 통계 | 사용 | 504행 triple_zero 검출 |
| ../Data/전입통계.csv | 전입 통계 | 사용 | 결측 0건 |
| ../Data/전출통계.csv | 전출 통계 | 사용 | 결측 0건 |
| ../Data/성남시_공시지가_통합__202604221844.csv | 공시지가 | 사용 | 산 필지 15행 → 제외 |
| ../Data/202301_202306_연령별인구현황_월간.csv | 주민등록 인구 (검증용) | 검증용 | 신용 0값 검증에 사용 |


In [ ]:
# 환경 설정 — 라이브러리 / 출력 옵션
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)
pd.options.display.float_format = '{:.0f}'.format

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')

# 작업 위치 기준 경로
PERSON_ROOT = Path.cwd()
DATA_DIR = PERSON_ROOT.parent / 'Data'
OUTPUT_DIR = PERSON_ROOT
print(f'PERSON_ROOT : {PERSON_ROOT}')
print(f'DATA_DIR    : {DATA_DIR}')
print(f'OUTPUT_DIR  : {OUTPUT_DIR}')


In [ ]:
# 사용할 원본 파일 목록
source_files = {
    'subway':   DATA_DIR / 'subway.csv',
    'bus':      DATA_DIR / 'bus_plz.csv',
    'credit':   DATA_DIR / '신용정보.csv',
    'in':       DATA_DIR / '전입통계.csv',
    'out':      DATA_DIR / '전출통계.csv',
    'price':    DATA_DIR / '성남시_공시지가_통합__202604221844.csv',
    'resident': DATA_DIR / '202301_202306_연령별인구현황_월간.csv',
}

for name, path in source_files.items():
    print(f'[{name}] {path} -> {"존재함" if path.exists() else "없음"}')


## 3. 데이터 로드

각 파일을 동일한 인코딩(`utf-8-sig` 또는 `cp949`) 규칙으로 불러온다.


In [ ]:
# 지하철
df_subway = pd.read_csv(source_files['subway'], encoding='utf-8-sig')
print('subway shape:', df_subway.shape)
display(df_subway.head(3))


In [ ]:
# 버스 (정류소번호 컬럼이 mixed type이라 경고 발생함 — 정상)
df_bus = pd.read_csv(source_files['bus'], encoding='utf-8-sig')
print('bus shape:', df_bus.shape)
display(df_bus.head(3))


In [ ]:
# 신용 / 전입 / 전출
df_credit = pd.read_csv(source_files['credit'], encoding='utf-8-sig')
df_in = pd.read_csv(source_files['in'], encoding='utf-8-sig')
df_out = pd.read_csv(source_files['out'], encoding='utf-8-sig')
print('credit shape:', df_credit.shape)
print('in     shape:', df_in.shape)
print('out    shape:', df_out.shape)


In [ ]:
# 공시지가
df_price = pd.read_csv(source_files['price'], encoding='utf-8-sig')
print('price shape:', df_price.shape)
display(df_price.head(3))


## 4. 데이터 기본 점검

각 데이터셋의 행/열, 결측치, 중복, 타입을 한 번에 확인한다.


In [ ]:
def quick_check(df, name):
    print(f'\n===== {name} =====')
    print('shape:', df.shape)
    print('\n[dtypes]')
    print(df.dtypes)
    print('\n[isnull sum > 0 only]')
    null_s = df.isnull().sum()
    print(null_s[null_s > 0] if null_s.sum() else '결측 없음')
    print('\n[duplicated count]:', df.duplicated().sum())

for name, df in [('subway', df_subway), ('bus', df_bus), ('credit', df_credit),
                 ('in', df_in), ('out', df_out), ('price', df_price)]:
    quick_check(df, name)


## 5. 문제 데이터 발견 목록 / 처리 기준 문서화

### 5-1. 문제 데이터 발견 목록 표

| 데이터셋 | 컬럼 | 문제 유형 | 발견 내용 | 처리 방식 | 이유 |
|---|---|---|---|---|---|
| 버스 | 행정동 | 결측 | 65,110건 행정동 NaN | 정류소ID 또는 정류소명 기반 매핑 보정 | 공간 키 누락 시 조인 불가 |
| 버스 | 행정동 | 중복 매핑 | 정류소ID 206000616, 206000617 — 동일 정류소가 백현동/삼평동 양쪽에 매핑 | 잘못된 행을 삭제 | 동별 합계가 부풀려짐 |
| 버스 | 정류소명 | 변경 | 2023년 ↔ 2024년 정류소명 변경 (성남역 개통 영향) | 통합 명칭으로 재할당 | 동일 위치 식별 |
| 버스 | 데이터 범위 | 성남 외 | 고기3리.유원지입구 등 용인 정류소 포함 | 행 삭제 | 분석 범위 외 |
| 신용 | TOT_CNT/ECON_CNT/NECON_CNT | 0값 (triple_zero) | 504행 / 운중동·고등동 집중 | structural_zero / error_zero 분류 후 분석에서 제외 (원본은 보존) | 임의 평균 대체 시 왜곡 |
| 공시지가 | 특수지구분명 | 산 필지 | 5필지 × 3년 = 15행 | 분석에서 제외 | 주거/상업 분석 대상 외 |
| 공시지가 | 공시지가 | 이상치 후보 | IQR 기준 1,168행 상위 이탈 | 제거하지 않고 boxplot/log 변환으로 점검만 수행 | 실제 강남급 도심 가격대 가능 |
| 지하철 | 전 컬럼 | 결측 | 0건 | 그대로 유지 | 처리 불필요 |
| 전입/전출 | 전 컬럼 | 결측 | 0건 | 그대로 유지 | 처리 불필요 |

### 5-2. 0값 처리 기준표 (신용 데이터)

| 분류 | 정의 | 처리 방식 |
|---|---|---|
| `normal` | 그룹 내에 0이 아닌 값이 함께 존재 | 그대로 사용 |
| `error_zero` | 같은 (월·BCD·연령·동) 그룹 안에 0과 비0이 공존 | 분석에서 제외 + 감사 로그 보관 |
| `structural_zero` | 그룹 전체가 모두 0 | 분석에서 제외 + 별도 점검 후 보고 |

### 5-3. 원본 유지 / 분석용 NaN 처리 원칙

- 원본 유지 원칙: 모든 raw csv는 절대 덮어쓰지 않는다. 분석용 사본(`clean_df`)에서만 처리한다.
- 분석용 처리 원칙: 임의 평균/중앙값 대체보다 `exclude_from_analysis` 플래그 또는 행 제거를 우선한다.
- 예외: 행정동 매핑은 명문화된 보정표에 한해 직접 치환한다.


## 6. 성주 전용 기준표

### 6-1. 정류소-행정동 보정표 (실제 적용)

| 정류소 ID | 정류소번호 | 연도 / 정류소명 | 기존 행정동 | 수정 행정동 | 수정 사유 |
|---|---|---|---|---|---|
| 206000305 | 7159 | 2023: 아름마을.이매고교.효성선경아파트.하나은행<br>2024: 성남역.이매고교.아름마을.효성선경아파트 | NaN | 이매2동 | 정류소명 변경 + 동일 위치 확인 |
| 206000314 | 7303 | 2023: 아름마을.이매고교.효성선경아파트.하나은행<br>2024: 성남역.이매고교.아름마을.효성선경아파트 | NaN | 이매2동 | 정류소명 변경 + 동일 위치 확인 |
| 206000530 | 7487 | 2023: 백현마을2단지<br>2024: 성남역.백현마을2단지 | NaN | 백현동 | 정류소명 변경 + 동일 위치 확인 |
| 206000537 | 7420 | 2023: 백현마을3단지<br>2024: 성남역.백현마을3단지 | NaN | 백현동 | 정류소명 변경 + 동일 위치 확인 |
| 206000616 | 7556 | 2023: 보평중고등학교<br>2024: 성남역.보평중고등학교 | 백현동, 삼평동 | 삼평동 유지 / 백현동 삭제 | 중복 매핑 제거 |
| 206000617 | 7557 | 2023: 보평중고등학교<br>2024: 성남역.보평중고등학교 | 백현동, 삼평동 | 백현동 유지 / 삼평동 삭제 | 중복 매핑 제거 |

> 그 외 정류소명 → 행정동 매핑 dictionary 4세트 (`dong_map`, `dong_map_update`, `dong_map_update2`, `dong_map_update3`, `dong_map_update4`) 적용. 모든 NaN 채움 후 잔여 결측은 0건.

### 6-2. 성남시 외 정류소 제외
- 고기3리.유원지입구 / 고기동마을 / 고기초등학교 / 왕재건설중기 → 행 삭제 (용인시 정류소)

### 6-3. 산 / 공원 / 비주거성 필지 제외 기준 (공시지가)

| 분류 | 위치 | 판단 | 처리 |
|---|---|---|---|
| 산 | 분당구 동원동 67-8 | 산지 | 삭제 |
| 산 | 수정구 창곡동 116-4 | 산지 | 삭제 |
| 공원 | 수정구 수진동 41-3 | 공원 | 삭제 |
| 산 안 시설 | 수정구 상적동 52-3 | 관리사무소 추정 | 삭제 |
| 임야상 주택 | 수정구 단대동 164-3 | 토지계획상 임야 | 삭제 |

> 분석 대상과 거리가 멀어 모두 삭제 처리.

### 6-4. 산번지 예외 검토 조건
- 실제 상업/주거 활용 흔적이 확인되는 경우
- 팀 분석 범위상 포함 근거가 분명한 경우
- 제외 시 표본 손실이 매우 크고 별도 표기 후 활용 가능한 경우


## 7. 전처리 실행 — 지하철

지하철 데이터는 결측·중복 모두 0이라 별도 처리 없이 저장한다.


In [ ]:
# 지하철 — clean = raw
df_subway_clean = df_subway.copy()

print('subway clean shape:', df_subway_clean.shape)
df_subway_clean.to_csv(OUTPUT_DIR / '2차전처리_지하철데이터.csv',
                       index=False, encoding='utf-8-sig')
print('[SAVE] 2차전처리_지하철데이터.csv')


## 8. 전처리 실행 — 버스

1) 정류소ID 단위 중복 매핑 제거 → 2) 정류소명 통일 → 3) 행정동 매핑 보정 → 4) 성남 외 정류소 삭제 → 5) 중복 제거 → 6) 월 단위 집계


In [ ]:
df_bus_clean = df_bus.copy()
df_bus_clean['승하차일자'] = pd.to_datetime(df_bus_clean['승하차일자'], errors='coerce')

# 1) 206000616 / 7556 — 백현동 행 삭제 (삼평동 유지)
mask_616 = (
    (pd.to_numeric(df_bus_clean['정류소ID'], errors='coerce') == 206000616) &
    (pd.to_numeric(df_bus_clean['정류소번호'], errors='coerce') == 7556)
)
df_bus_clean = df_bus_clean[~(mask_616 & (df_bus_clean['행정동'] == '백현동'))]

mask_616 = (
    (pd.to_numeric(df_bus_clean['정류소ID'], errors='coerce') == 206000616) &
    (pd.to_numeric(df_bus_clean['정류소번호'], errors='coerce') == 7556)
)
df_bus_clean.loc[mask_616 & (df_bus_clean['승하차일자'].dt.year == 2023) &
                 (df_bus_clean['정류소명'].isna()), '정류소명'] = '보평중고등학교'
df_bus_clean.loc[mask_616 & (df_bus_clean['승하차일자'].dt.year == 2024) &
                 (df_bus_clean['정류소명'].isna()), '정류소명'] = '성남역.보평중고등학교'

# 2) 206000617 / 7557 — 삼평동 행 삭제 (백현동 유지)
mask_617 = (
    (pd.to_numeric(df_bus_clean['정류소ID'], errors='coerce') == 206000617) &
    (pd.to_numeric(df_bus_clean['정류소번호'], errors='coerce') == 7557)
)
df_bus_clean = df_bus_clean[~(mask_617 & (df_bus_clean['행정동'] == '삼평동'))]

mask_617 = (
    (pd.to_numeric(df_bus_clean['정류소ID'], errors='coerce') == 206000617) &
    (pd.to_numeric(df_bus_clean['정류소번호'], errors='coerce') == 7557)
)
df_bus_clean.loc[mask_617 & (df_bus_clean['승하차일자'].dt.year == 2023) &
                 (df_bus_clean['정류소명'].isna()), '정류소명'] = '보평중고등학교'
df_bus_clean.loc[mask_617 & (df_bus_clean['승하차일자'].dt.year == 2024) &
                 (df_bus_clean['정류소명'].isna()), '정류소명'] = '성남역.보평중고등학교'


In [ ]:
# 3) 정류소명 / 행정동 고정 매핑 (성남역 개통 관련)
name_map = {
    206000305: '성남역.이매고교.아름마을.효성선경아파트',
    206000314: '성남역.이매고교.아름마을.효성선경아파트',
    206000530: '성남역.백현마을2단지',
    206000537: '성남역.백현마을3단지',
}
for stop_id, new_name in name_map.items():
    df_bus_clean.loc[df_bus_clean['정류소ID'] == stop_id, '정류소명'] = new_name

dong_fix_map = {
    206000305: '이매2동',
    206000314: '이매2동',
    206000530: '백현동',
    206000537: '백현동',
}
for stop_id, dong in dong_fix_map.items():
    df_bus_clean.loc[df_bus_clean['정류소ID'] == stop_id, '행정동'] = dong


In [ ]:
# 4) 성남시 외 정류소 삭제
df_bus_clean = df_bus_clean[
    ~df_bus_clean['정류소명'].isin([
        '고기3리.유원지입구', '고기동마을', '고기초등학교', '왕재건설중기',
    ])
]

# 5) 정류소명 → 행정동 매핑 사전 (4세트 통합)
dong_map_all = {
    # set 1
    '(임시)금토동삼거리': '금토동', 'SK.금강테라스하우스': '대장동',
    '고등동우체국.성남농협대왕지점': '고등동', '고등동행정복지센터.성남농협대왕지점': '고등동',
    '공단치안센터.근로자종합복지관': '상대원동', '곽여성병원(마을)': '태평동',
    '교보문고(마을)': '백현동', '구미공원.불곡초등학교': '구미동', '구미동행정복지센터': '구미동',
    '구서고.금상초교': '금광동', '국은교앞': '운중동',
    '금상초교.이편한세상3단지': '금광동', '금상초교.이편한세상5단지': '금광동',
    '나라기록관.코이카': '시흥동', '남한산성공원입구': '양지동', '남한산성공원입구.양지파출소': '양지동',
    '노블빌리지': '구미동', '농수산물센터사거리': '구미동', '농수산물센터사거리.LG트윈하우스': '구미동',
    '단대오거리역.성남시박물관': '신흥동', '단대오거리역.세이브존.성남시박물관': '신흥동',
    '대하초등학교.중원유스센터': '하대원동', '더블트리바이힐튼호텔.잡월드': '정자동',
    '동국대한방병원.신성아파트.수내도서관': '수내동', '동원동종점.모드니': '동원동',
    '목련마을SK아파트.성남시차량등록사업소': '야탑동', '무지개도서관.구미파출소.건영아파트': '구미동',
    '무지개마을사거리': '구미동', '백현동종점': '백현동',
    # set 2
    '다이소앞': '구미동', '벽산아파트': '수내동', '보평중고등학교': '백현동',
    '봇들마을7.8단지.롯데마트판교점': '삼평동', '분당경영고.한라아파트': '금곡동',
    '분당아테나': '수내동', '분당아테라': '수내동',
    '사송동.한국수자원공사': '사송동', '사송동.한국수자원공사(마을)': '사송동',
    '상대원119안전센터.성남공단우체국': '상대원동', '새마을연수원입구': '율동',
    '서현유스센터.양영디지털고등학교': '서현동',
    '선경아파트.성남중원경찰서.중원구보건소': '상대원동', '선텍시티2차': '상대원동',
    '성남고등공공주택지구.동편': '고등동', '성남고등공공주택지구.서편': '고등동',
    '성남금융고': '야탑동', '성남금융고.중탑초교': '야탑동',
    '성남냉동': '하대원동', '성남농협대왕지점.고등동우체국': '고등동',
    '성남물빛정원남측.노블빌리지': '구미동', '성남물빛정원서측.한국토지주택공사': '구미동',
    '성남북초등학교.산성동행정복지센터.산성역자이4단지': '산성동', '성남시근로자종합복지관': '상대원동',
    '성남시여성비전센터.삼성생명.한국사회적기업진흥원': '태평동',
    '성남시여성비전센터.성남수정새마을금고.한국사회적기업진흥원': '태평동',
    '성남시의료원.신흥1동행정복지센터.성남아트리움': '신흥동',
    '성남시장례문화사업소': '갈현동', '성남시장례문화사업소.장례식장입구': '갈현동',
    '성남역': '백현동',
    # set 3
    '성남역.보평중고등학교': '백현동', '성남중원경찰서': '상대원동',
    '성남혜은학교.수정유스센터.산성역자이3단지': '신흥동', '성보경영고등학교': '단대동',
    '세이브존.성남시박물관': '신흥동', '셀레스빌아파트.포레스티아중문': '신흥동',
    '수내2동행정복지센터': '수내동', '수내고등학교.수내도서관': '수내동',
    '수정구청.산성역자이2단지': '신흥동', '수정구청.포레스티아서문': '신흥동',
    '수정초등학교': '수진동', '수진동성당': '수진동',
    '순환도시친환경세상순환자원홍보관': '석운동', '쉐보레분당서비스센터': '금곡동',
    '시흥동농협창고': '시흥동', '시흥사거리.고등공공주택지구': '시흥동',
    '신지교회': '성남동', '아튼빌아파트.중원유스센터': '하대원동',
    '에스콰이어': '상대원동', '옛근로자종합복지관': '성남동',
    '오리삼거리(마을)': '구미동', '우성아파트': '정자동',
    '위례31단지.위례포레샤인후문': '창곡동', '위례31단지정문': '창곡동',
    '위례동행정복지센터.위례보미리즌빌': '창곡동', '위례동행정복지센터.위례아트리버푸르지오': '창곡동',
    '위례자연앤래미안e편한세상.위례한빛중학교': '창곡동',
    '위례자연앤래미안e편한세상.위례힐스테이트': '창곡동',
    '위례자이아파트': '창곡동',
    # set 4
    '위례자이후문': '창곡동', '위례중앙초등학교': '창곡동',
    '위례포레샤인.위례31단지후문': '창곡동', '위례한빛고등학교': '창곡동',
    '위례한빛초.래미안.힐스테이트.한빛마을': '창곡동',
    '위례한빛초교.힐스테이트후문.한빛마을': '창곡동',
    '위례호반베르디움': '창곡동', '위례힐스테이트정문': '창곡동',
    '위례힐스테이트후문.위례한빛초등학교': '창곡동',
    '이노밸리.포스코ICT': '삼평동', '이매역.진흥아파트.동신아파트': '이매동',
    '이우중고등학교.주성카센타': '동원동', '이지더원.삼평동행정복지센터': '삼평동',
    '이편한세상5단지': '금광동', '이편한세상금빛그랑메종': '금광동',
    '자연앤센트럴자이': '창곡동', '자연앤센트럴자이정문': '창곡동',
    '정자유스센터.분당클리닉': '정자동', '주공12단지': '구미동',
    '주공7단지앞.치매안심센터': '정자동',
    '주공7단지앞.한솔종합사회복지관.치매안심센터': '정자동',
    '중동고개.제일초교': '중앙동', '중앙동사거리': '중앙동',
    '중원구청.성남소방서': '성남동', '진로아파트': '단대동',
    '창조밸리교': '금곡동', '창조밸리교(경유)': '금곡동',
    '청솔마을.계룡아파트': '금곡동', '청솔마을.화인아파트': '금곡동',
    '파크뷰아파트.정자유스센터': '정자동',
    # set 5
    '판교고.송현초.삼평동행정복지센터': '삼평동', '판교대장초?중학교': '대장동',
    '판교더샵포레스트11단지': '대장동', '판교청소년수련관.판교종합사회복지관': '판교동',
    '판교퍼스트힐푸르지오2단지': '대장동',
    '포레스티아서문.신흥2동행정복지센터.산성역자이2단지': '신흥동',
    '푸른마을신성아파트.수내도서관': '수내동',
    '하대원동행정복지센터.영성중.검단초': '하대원동',
    '한국폴리텍대학.성남문화예술교육센터': '산성동',
    '한양수자인성남마크뷰.황성게이트볼장': '금광동',
    '헬스케어혁신파크.(구)가스공사': '정자동',
    '현대노블리스': '구미동', '힐스테이트판교엘포레6단지후문': '대장동',
}

df_bus_clean['행정동'] = df_bus_clean['행정동'].fillna(
    df_bus_clean['정류소명'].map(dong_map_all)
)
print('잔여 행정동 결측:', df_bus_clean['행정동'].isna().sum())
print('잔여 정류소명 결측:', df_bus_clean['정류소명'].isna().sum())


In [ ]:
# 6) 중복 제거 + 월 단위로 집계
print('중복 행 수 (제거 전):', df_bus_clean.duplicated().sum())
df_bus_clean = df_bus_clean.drop_duplicates()
print('중복 행 수 (제거 후):', df_bus_clean.duplicated().sum())

df_bus_clean['승하차일자'] = pd.to_datetime(df_bus_clean['승하차일자']).dt.to_period('M')
print('bus clean shape:', df_bus_clean.shape)

df_bus_clean.to_csv(OUTPUT_DIR / '2차전처리_버스데이터.csv',
                    index=False, encoding='utf-8-sig')
print('[SAVE] 2차전처리_버스데이터.csv')


## 9. 전처리 실행 — 신용 (대민개방)

`TOT_CNT == 0 & ECON_CNT == 0 & NECON_CNT == 0` 인 504행을 **structural_zero / error_zero** 로 분류하고, 분석용 사본에서만 제외한다. 원본은 보존.

추가로 2023년 상반기 구간은 `202301_202306_연령별인구현황_월간.csv` 로 검증 → 운중동의 0값 대부분이 `주민등록상 인구 있음 → 신용 0 재점검 필요` 로 판정됨.


In [ ]:
# 신용 데이터 0값 분류
df = df_credit.copy()
key_cols = ['BS_YR_MON', 'BCD', 'AGE', '읍면동명', '시군구명']

df['triple_zero'] = (
    (df['TOT_CNT'] == 0) &
    (df['ECON_CNT'] == 0) &
    (df['NECON_CNT'] == 0)
)

grp = df.groupby(key_cols).agg(
    row_cnt=('TOT_CNT', 'size'),
    all_triple_zero=('triple_zero', 'all'),
    any_triple_zero=('triple_zero', 'any'),
    any_nonzero_tot=('TOT_CNT', lambda x: (x > 0).any()),
).reset_index()

def classify_zero(row):
    if row['any_triple_zero'] and row['any_nonzero_tot']:
        return 'error_zero'
    if row['all_triple_zero']:
        return 'structural_zero'
    return 'normal'

grp['zero_type'] = grp.apply(classify_zero, axis=1)

df2 = df.merge(grp[key_cols + ['zero_type']], on=key_cols, how='left')
df2['exclude_from_analysis'] = (
    df2['triple_zero'] &
    df2['zero_type'].isin(['error_zero', 'structural_zero'])
)

print('zero_type 분포 (그룹 기준):')
print(grp['zero_type'].value_counts())

print('\nexclude_from_analysis 분포 (행 기준):')
print(df2['exclude_from_analysis'].value_counts())

# 분석용 데이터 + 감사 로그 보관
clean_df = df2[~df2['exclude_from_analysis']].copy()
zero_log = df2[df2['triple_zero']].copy()

clean_df = clean_df.rename(columns={'읍면동명': '행정동'})
print('\nclean_df shape:', clean_df.shape, '/ 원본 shape:', df.shape)

clean_df.to_csv(OUTPUT_DIR / '2차전처리_대민개방데이터.csv',
                index=False, encoding='utf-8-sig')
print('[SAVE] 2차전처리_대민개방데이터.csv')


### 9-1. 주민등록 데이터로 신용 0값 검증 (2023년 상반기)


In [ ]:
# 주민등록 월간 인구 데이터로 신용 0값 검증
resident_path = source_files['resident']

resident_monthly = None
for enc in ['cp949', 'euc-kr', 'utf-8-sig']:
    try:
        resident_monthly = pd.read_csv(resident_path, encoding=enc)
        print(f'주민등록 월간 파일 로드 / encoding = {enc}')
        break
    except Exception:
        pass

first_col = resident_monthly.columns[0]
resident_monthly = resident_monthly.rename(columns={first_col: '행정구역'})
resident_monthly['resident_code'] = resident_monthly['행정구역'].astype(str).str.extract(r'\((\d{10})\)', expand=False)
resident_monthly['읍면동명'] = resident_monthly['행정구역'].astype(str).str.extract(r'([가-힣]+)\(\d{10}\)$', expand=False)

target_age_map = {
    '18~29세': 20, '30~39세': 30, '40~49세': 40,
    '50~59세': 50, '60~69세': 60, '70~79세': 70,
}
target_cols = [
    col for col in resident_monthly.columns
    if any(b in col for b in target_age_map.keys())
    and ('_거주자_' in col)
    and ('남_' not in col) and ('여_' not in col)
]

resident_long = resident_monthly.melt(
    id_vars=['행정구역', 'resident_code', '읍면동명'],
    value_vars=target_cols,
    var_name='resident_col',
    value_name='resident_pop_raw'
)
resident_long['월'] = resident_long['resident_col'].str.extract(r'2023년(\d{2})월', expand=False)
resident_long['BS_YR_MON'] = ('2023' + resident_long['월']).astype(int)
resident_long['age_band'] = resident_long['resident_col'].str.extract(
    r'(20~29세|30~39세|40~49세|50~59세|60~69세|70~79세)', expand=False)
resident_long['AGE'] = resident_long['age_band'].map(target_age_map)
resident_long['resident_pop'] = (
    resident_long['resident_pop_raw'].astype(str)
    .str.replace(',', '', regex=False).astype(int)
)
resident_long = resident_long[['행정구역', 'resident_code', '읍면동명', 'BS_YR_MON', 'AGE', 'resident_pop']]

credit_zero_h1 = df_credit[
    (df_credit['TOT_CNT'] == 0) & (df_credit['ECON_CNT'] == 0) & (df_credit['NECON_CNT'] == 0) &
    (df_credit['BS_YR_MON'].between(202301, 202306))
].copy()

compare_zero_h1 = credit_zero_h1.merge(
    resident_long, on=['읍면동명', 'BS_YR_MON', 'AGE'], how='left'
)
compare_zero_h1['resident_check_result'] = np.select(
    [
        compare_zero_h1['resident_pop'].isna(),
        compare_zero_h1['resident_pop'] > 0,
        compare_zero_h1['resident_pop'] == 0,
    ],
    [
        '매칭불가',
        '주민등록상 인구 있음 → 신용 0 재점검 필요',
        '주민등록상도 0 가능',
    ],
    default='확인 필요',
)

print('[2023년 1~6월 검증 결과]')
display(compare_zero_h1['resident_check_result'].value_counts().rename_axis('판정').reset_index(name='건수'))


## 10. 전처리 실행 — 전입 / 전출

전입·전출 통계는 결측·이상치 점검 결과 모두 0이라 그대로 저장한다.


In [ ]:
df_in_clean = df_in.copy()
df_out_clean = df_out.copy()

print('in shape:', df_in_clean.shape, ' / out shape:', df_out_clean.shape)

df_in_clean.to_csv(OUTPUT_DIR / '2차전처리_전입데이터.csv',
                   index=False, encoding='utf-8-sig')
df_out_clean.to_csv(OUTPUT_DIR / '2차전처리_전출데이터.csv',
                    index=False, encoding='utf-8-sig')
print('[SAVE] 2차전처리_전입데이터.csv')
print('[SAVE] 2차전처리_전출데이터.csv')


## 11. 전처리 실행 — 공시지가

1) 산 필지 5건(× 3년 = 15행) 제외
2) 법정동명에서 구(분당/수정/중원) 추출
3) 기준연도 2023~2025, 기준월 1·7월(연 2회) 확인
4) IQR / boxplot / log 변환으로 이상치 점검 (제거하지 않음)


In [ ]:
df_price_clean = df_price.copy()

print('산 필지 (제거 전):')
display(df_price_clean[df_price_clean['특수지구분명'] == '산'])

# 산 필지 제거
df_price_clean = df_price_clean[df_price_clean['특수지구분명'] != '산']

# 구 컬럼 추가
df_price_clean['구'] = df_price_clean['법정동명'].str.extract(r'(분당구|수정구|중원구)')
print('\n구별 행 수:')
print(df_price_clean['구'].value_counts())


In [ ]:
# 시점 점검
print('기준연도:', sorted(df_price_clean['기준연도'].unique()))
print('기준월  :', sorted(df_price_clean['기준월'].unique()))
print('기준년월:', sorted(df_price_clean['기준년월'].unique()))

# 공시지가 분포
print('\n공시지가 describe:')
display(df_price_clean['공시지가'].describe())

# IQR 이상치 후보 (제거하지 않음 — 단순 점검)
q1 = np.percentile(df_price_clean['공시지가'], 25)
q3 = np.percentile(df_price_clean['공시지가'], 75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
outliers = df_price_clean[(df_price_clean['공시지가'] < lower) |
                          (df_price_clean['공시지가'] > upper)]
print(f'\nIQR 이상치 후보 (제거 X, 점검만): {len(outliers)}행')


In [ ]:
# Boxplot 점검 (원본 vs log1p 변환)
df_price_clean['공시지가_log'] = np.log1p(df_price_clean['공시지가'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].boxplot(df_price_clean['공시지가'].dropna())
axes[0].set_title('공시지가 Boxplot')
axes[0].set_ylabel('공시지가 (원/㎡)')
axes[1].boxplot(df_price_clean['공시지가_log'].dropna())
axes[1].set_title('공시지가 Log Boxplot')
axes[1].set_ylabel('log(공시지가)')
plt.tight_layout()
plt.show()

df_price_clean.to_csv(OUTPUT_DIR / '2차전처리_공시지가데이터.csv',
                      index=False, encoding='utf-8-sig')
print('[SAVE] 2차전처리_공시지가데이터.csv')


## 12. clean 파일 저장 전후 비교

각 데이터셋의 처리 전/후 행 수 변화와 처리 원칙을 정리한다.

| 데이터셋 | before shape | after shape | 변경 내용 |
|---|---|---|---|
| subway | (850, 5) | (850, 5) | 변경 없음 (결측 0건) |
| bus | (2,058,750, 10) | 약 (1,297,105, 10) | 정류소-행정동 보정 + 중복 제거 + 월 집계 |
| credit | (29,818, 9) | 약 (29,332, 12) | triple_zero 504행 분석 제외 + 컬럼 3개 추가 |
| in(전입) | (405,746, 31) | (405,746, 31) | 변경 없음 |
| out(전출) | (402,335, 31) | (402,335, 31) | 변경 없음 |
| price(공시지가) | (15,557, 11) | (15,542, 12) | 산 필지 15행 제외 + 구 컬럼 추가 |

### 최종 처리 원칙 요약

- **0값**은 실제 0인지 결측 대체값인지 먼저 확인하고, 원본은 유지한 채 분석용 사본에서만 별도 처리한다.
- **정류소-행정동 불일치**는 보정표를 별도로 작성하고, 근거 없는 일괄 치환은 하지 않는다.
- **공시지가의 산 필지**는 분석 대상 외이므로 제외하되, 기준은 "특수지구분명 == '산'" 으로 명문화한다.
- **신용 데이터의 triple_zero**는 임의 대체보다 `exclude_from_analysis` 플래그 + 감사 로그 보관 원칙을 우선한다.
- **이상치(공시지가)**는 IQR 기준만으로 제거하지 않는다. 도심 고가 필지를 정상값으로 인정하기 위함.


## 13. 교통 EDA 핵심 결과 요약

`EDA_교통.ipynb` 에서 도출한 핵심 해석을 한 페이지로 정리.

### 13-1. 시계열 패턴
- **2024년 10월 (≒ 2024년 3분기)** 이 행정동·역별 이용량의 **구조적 변화 시점**으로 관측됨 → 정책/노선 변경/외부 이벤트 가능성.
- 행정동 간 격차는 계속 유지(상위는 계속 상위, 하위는 계속 하위), 전체 수요는 증가 추세.

### 13-2. 로그 변화율 분석 (2023-01 vs 최종월 / 분기)
- 양수(파랑) = 외부 소비자 유입 증가 → **임대료 상승 압력 높은 지역**
- 음수(빨강) = 소비자 이탈 → **소상공인 이탈 가속 가능성**
- 1,000 미만 소규모 행정동은 통계적 왜곡 방지를 위해 제외.

### 13-3. HHI(허핀달-허시만) 기반 정류소 집중도
- HHI 높음 → 대형 허브에 집중 → 골목 상권 집객력 약화 → 폐업 압력
- HHI 낮음 → 다수 정류소 분산 → 골목 상권/전통시장 살아있음

### 13-4. 산점도 — 소비 유입 × 집중도 사분면

| 사분면 | 이용량 | HHI | 해석 |
|---|---|---|---|
| 우상단 | 증가 | 높음 | **위험 의심** — 소비자 늘었지만 대형 허브에만 몰림 |
| 우하단 | 증가 | 낮음 | 골목 상권 전반 활성화 — 양극화 없음 |
| 좌상단 | 감소 | 높음 | 소규모 상권 붕괴 완료 — 대형 허브만 잔존 |
| 좌하단 | 감소 | 낮음 | 상권 전반 쇠퇴 — 소비자 자체 감소 |

### 13-5. 정류소명 키워드 기반 상업형 / 생활형 분류
- 상업형 키워드: 역, 터미널, 백화점, 플라자, 몰, 아울렛, 마트, 쇼핑, 시장
- 생활형 키워드: 아파트, 학교, 병원, 행정복지센터 등
- **상업형 비중 높은 동** → 외부 소비 유입 구조 → 임대료 상승·소상공인 이탈 압력 높음
- **생활형 비중 높은 동** → 골목 상권 중심, 상업화 압력 낮음

### 13-6. 지하철 핵심
- 야탑·신흥·가천대: 하차 우세 → 외부 유입 중심 → 상업 기능 강화 가능성
- 수내·서현·정자: 승차 우세 → 주거/통근 중심 → 소비 유출 구조
- 경강선/신분당선 HHI 높음(집중형), 8호선/분당선 HHI 낮음(분산형)
- 우상단(하차 증가 + 하차비율 > 0.5) 역 = 젠트리피케이션 압력 가장 높은 후보


## 14. 최종 체크리스트

- [x] 문제 데이터 발견 목록 표 작성
- [x] 정류소-행정동 보정표 작성 (정류소ID 단위)
- [x] 산 / 공원 / 비주거성 필지 제외 기준 작성
- [x] 산번지 예외 검토 조건 정리
- [x] 신용 0값 분류 기준(structural_zero / error_zero) 작성
- [x] 주민등록 데이터 기반 신용 0값 교차 검증
- [x] 원본 유지 / 분석용 NaN 처리 원칙 문서화
- [x] clean 파일 저장 전후 비교
- [x] 교통 EDA 핵심 결과 요약 (시계열·HHI·사분면·상업/생활 분류)
- [x] 6개 clean csv 저장 완료


## 15. 팀 공유용 5줄 요약

1. 교통(지하철·버스), 신용(대민·전입·전출), 공시지가 6종 데이터의 1차/2차 전처리 결과를 정리하고 예외 처리 기준을 모두 표로 명문화했다.
2. 버스 데이터에서 행정동 결측 65,110건과 정류소ID 단위 중복 매핑(206000616/617)을 보정표 기반으로 처리해 잔여 결측 0건을 달성했다.
3. 신용 데이터의 504행 triple_zero는 `structural_zero / error_zero` 로 분류해 분석에서 제외했고, 운중동 0값은 주민등록 인구와 매칭하여 "신용 0 재점검 필요"로 판정했다.
4. 공시지가에서는 산 필지 15행만 제외하고 IQR 이상치는 도심 고가 필지를 보존하기 위해 제거하지 않았다.
5. 교통 EDA에서 2024년 3·4분기를 구조적 변화 시점으로 식별하고, HHI×하차 로그 변화율 사분면으로 젠트리피케이션 위험 지역 후보를 분류할 수 있는 분석 틀을 정리했다.
